# Google Gemini

## Simple Variable Substitution

In [1]:
from langchain_core.prompts import PromptTemplate

# Create a simple template
template = """You are a helpful assistant that translates {input_language} to {output_language}.

Text: {text}

Translation:"""

# Create the prompt template
prompt = PromptTemplate(
    input_variables=["input_language", "output_language", "text"],
    template=template
)

# Use the template
formatted_prompt = prompt.format(
    input_language="English",
    output_language="Spanish",
    text="Hello, how are you?"
)

print(formatted_prompt)

# Output will be:
# You are a helpful assistant that translates English to Spanish.
#
# Text: Hello, how are you?
#
# Translation:

You are a helpful assistant that translates English to Spanish.

Text: Hello, how are you?

Translation:


## Complete Example: Template + LLM

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create the template
template = """You are a helpful assistant that translates {input_language} to {output_language}.

Text: {text}

Translation:"""

prompt = PromptTemplate(
    input_variables=["input_language", "output_language", "text"],
    template=template
)

# Initialize the LLM with API key
# Make sure you have GOOGLE_API_KEY in your .env file
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Create a chain: template -> LLM
chain = prompt | model

# Run the chain (this will actually translate!)
response = chain.invoke({
    "input_language": "English",
    "output_language": "Spanish",
    "text": "Hello, how are you?"
})

print(response.content)
# Output: "Hola, ¿cómo estás?"

Hola, ¿cómo estás?


## Chat Prompt Templates

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Create a chat template with system and human messages
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} that helps with {task}."),
    ("human", "{user_input}"),
])

# Format the template
messages = chat_template.format_messages(
    role="programming tutor",
    task="Python coding",
    user_input="How do I read a CSV file?"
)

# Use with your model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")  # Set API key explicitly
)
response = llm.invoke(messages)
print(response.content)

Reading CSV (Comma Separated Values) files in Python is a very common task! There are two primary ways to do it, depending on your needs:

1.  **Using the `pandas` library:** This is the most popular and powerful method, especially if you're doing any kind of data analysis, manipulation, or working with larger datasets.
2.  **Using Python's built-in `csv` module:** This is a good choice if you want to avoid external libraries, need fine-grained control over row-by-row processing, or are working with very simple CSVs.

Let's look at both!

First, you'll need a sample CSV file. Let's create one named `data.csv`:

```csv
Name,Age,City,Occupation
Alice,30,New York,Engineer
Bob,24,London,Designer
Charlie,35,Paris,Doctor
Diana,28,Berlin,Artist
```

---

## 1. Using the `pandas` library (Recommended for Data Analysis)

If you don't have `pandas` installed, open your terminal or command prompt and run:
`pip install pandas`

```python
import pandas as pd

# Define the path to your CSV file
file

# Advanced Template Patterns


## Partial Templates

In [5]:
from langchain_core.prompts import PromptTemplate
from datetime import datetime

# Template with multiple variables
template = """Date: {date}
User: {user_name}
Task: {task}

Please complete the following: {request}"""

# Create a partial template with date pre-filled
prompt = PromptTemplate(
    input_variables=["user_name", "task", "request"],
    template=template,
    partial_variables={"date": datetime.now().strftime("%Y-%m-%d")}
)

# Now you only need to provide the remaining variables
formatted = prompt.format(
    user_name="Alice",
    task="Code Review",
    request="Review this Python function for best practices"
)

print(formatted)

Date: 2026-05-22
User: Alice
Task: Code Review

Please complete the following: Review this Python function for best practices


## Template Composition

In [6]:
from langchain_core.prompts import PromptTemplate

# Define reusable template components
persona_template = "You are a {expertise} expert with {years} years of experience."
task_template = "Your task is to {action} the following {content_type}:"
format_template = "Please format your response as {format}."

# Combine templates
full_template = f"""
{persona_template}

{task_template}

{format_template}

Content: {{content}}
"""

# Create the prompt
prompt = PromptTemplate(
    input_variables=["expertise", "years", "action", "content_type", "format", "content"],
    template=full_template
)

# Use the composed template
result = prompt.format(
    expertise="Python",
    years="10",
    action="optimize",
    content_type="code",
    format="a list of improvements with explanations",
    content="def calculate_sum(numbers): total = 0; for n in numbers: total = total + n; return total"
)

print(result)


You are a Python expert with 10 years of experience.

Your task is to optimize the following code:

Please format your response as a list of improvements with explanations.

Content: def calculate_sum(numbers): total = 0; for n in numbers: total = total + n; return total



# Output Parsers

**Why Output Parsers?**
LLMs return unstructured text, but applications need structured data. Output parsers convert free-form text into usable formats like:

## List Output Parser

Parse comma-separated or numbered lists. This parser automatically instructs the LLM to return results in a comma-separated format and then splits the response into a Python list.

In [8]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize parser
list_parser = CommaSeparatedListOutputParser()

# Create prompt with format instructions
prompt = PromptTemplate(
    template="List 5 {category}.\n{format_instructions}",
    input_variables=["category"],
    partial_variables={"format_instructions": list_parser.get_format_instructions()}
)

# Create chain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
chain = prompt | llm | list_parser

# Get parsed list
result = chain.invoke({"category": "Python web frameworks"})
print(result)  # ['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']

['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']


## Structured Output Parser

Parse complex structured data with validation. This parser uses ResponseSchema objects to define the expected structure and automatically instructs the LLM to return data in JSON format.

In [1]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Define the expected structure
response_schemas = [
    ResponseSchema(
        name="name",
        description="The name of the product"
    ),
    ResponseSchema(
        name="price",
        description="The price in USD as a number"
    ),
    ResponseSchema(
        name="features",
        description="A list of key features"
    ),
    ResponseSchema(
        name="in_stock",
        description="Whether the item is in stock (true/false)"
    )
]

# Create parser
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Create prompt
prompt = PromptTemplate(
    template="""Extract product information from the following text.

{format_instructions}

Text: {product_description}""",
    input_variables=["product_description"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

# Use the chain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
chain = prompt | llm | output_parser

# Example usage
result = chain.invoke({
    "product_description": "The new iPhone 15 Pro costs $999 and features a titanium design, A17 Pro chip, and improved camera system. Currently available in stores."
})

print(result)
# {'name': 'iPhone 15 Pro', 'price': 999, 'features': ['titanium design', 'A17 Pro chip', 'improved camera system'], 'in_stock': True}

{'name': 'iPhone 15 Pro', 'price': '999', 'features': 'titanium design, A17 Pro chip, improved camera system', 'in_stock': 'true'}


## Pydantic Output Parser
For production applications, use Pydantic for type safety and validation. Pydantic provides the most robust parsing with automatic type conversion, validation, and clear error messages.

🔍 **Why Pydantic Output Parser?**
- Type Safety: Automatic type checking and conversion
- Validation: Built-in field validation with custom rules
- Error Handling: Clear error messages when parsing fails
- IDE Support: Full autocomplete and type hints
- Production Ready: Battle-tested in enterprise applications

💡 **Key Components**:
- BaseModel: Define your data structure with types
- Field(): Add descriptions and validation rules
- PydanticOutputParser: Converts LLM output to typed objects
- format_instructions: Auto-generated prompt instructions

### Example: Product Information Extraction with Pydantic

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
from dotenv import load_dotenv

load_dotenv()

# --- Define output schema with Pydantic ---
class Product(BaseModel):
    name: str = Field(description="The name of the product")
    price: float = Field(description="The price in USD as a number")
    features: List[str] = Field(description="A list of key features")
    in_stock: bool = Field(description="Whether the item is in stock")

# --- Parser ---
output_parser = PydanticOutputParser(pydantic_object=Product)

# --- Prompt ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product data extractor. Extract structured product information accurately."),
    ("human", """Extract product information from the text below.

{format_instructions}

Text: {product_description}""")
]).partial(format_instructions=output_parser.get_format_instructions())

# --- Model ---
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0  # Deterministic output for structured extraction
)

# --- Chain ---
chain = prompt | llm | output_parser

# --- Run ---
result: Product = chain.invoke({
    "product_description": (
        "The new iPhone 15 Pro costs $999 and features a titanium design, "
        "A17 Pro chip, and improved camera system. Currently available in stores."
    )
})

print(result)
# Product(name='iPhone 15 Pro', price=999.0, features=['titanium design', 'A17 Pro chip', 'improved camera system'], in_stock=True)

# Access fields directly with full type safety
print(f"Name:     {result.name}")
print(f"Price:    ${result.price:.2f}")
print(f"Features: {', '.join(result.features)}")
print(f"In stock: {result.in_stock}")

name='iPhone 15 Pro' price=999.0 features=['titanium design', 'A17 Pro chip', 'improved camera system'] in_stock=True
Name:     iPhone 15 Pro
Price:    $999.00
Features: titanium design, A17 Pro chip, improved camera system
In stock: True


In [3]:
result

Product(name='iPhone 15 Pro', price=999.0, features=['titanium design', 'A17 Pro chip', 'improved camera system'], in_stock=True)

### Example: Recipe Extraction with Pydantic

In [6]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from typing import List
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Define your data model
class Recipe(BaseModel):
    name: str = Field(description="Name of the recipe")
    prep_time: int = Field(description="Preparation time in minutes")
    servings: int = Field(description="Number of servings")
    ingredients: List[str] = Field(description="List of ingredients")
    difficulty: str = Field(description="Difficulty level: easy, medium, or hard")

# Create parser
parser = PydanticOutputParser(pydantic_object=Recipe)

# Create prompt
prompt = PromptTemplate(
    template="""Extract recipe information from the following text.

{format_instructions}

Text: {recipe_text}""",
    input_variables=["recipe_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Use the chain
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
chain = prompt | llm | parser

# Parse recipe
result = chain.invoke({
    "recipe_text": "This easy pasta carbonara takes just 20 minutes to make and serves 4 people. You'll need spaghetti, eggs, bacon, parmesan cheese, and black pepper."
})

print(f"Recipe: {result.name}")
print(f"Time: {result.prep_time} minutes")
print(f"Servings: {result.servings}")
print(f"Ingredients: {', '.join(result.ingredients)}")
print(f"Difficulty: {result.difficulty}")

Recipe: pasta carbonara
Time: 20 minutes
Servings: 4
Ingredients: spaghetti, eggs, bacon, parmesan cheese, black pepper
Difficulty: easy


In [7]:
result

Recipe(name='pasta carbonara', prep_time=20, servings=4, ingredients=['spaghetti', 'eggs', 'bacon', 'parmesan cheese', 'black pepper'], difficulty='easy')